# 01 — Análisis exploratorio del consumo energético

**Problema:** predecir el consumo de electrodomésticos (`Appliances`, Wh).

**Dataset:** [UCI Appliances Energy Prediction](https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction) — `data/raw/energydata_complete.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

TARGET = "Appliances"
DATE_COL = "date"
DROP_COLS = ["rv1", "rv2"]
TEST_RATIO = 0.2
RANDOM_STATE = 42
USE_GRADIENT_BOOSTING = True

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "energydata_complete.csv"
FIGURES_DIR = ROOT / "reports" / "figures"
METRICS_DIR = ROOT / "reports" / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Coloca energydata_complete.csv en {DATA_PATH}")

In [ ]:
def load_and_clean(csv_path: Path) -> pd.DataFrame:
    """Carga el CSV UCI y aplica limpieza básica."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include="object").columns:
        if col != DATE_COL:
            df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    df = df.drop_duplicates()
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Variables temporales, climáticas y de rezago (solo información pasada)."""
    out = df.copy()
    dt = out[DATE_COL]

    out["hour"] = dt.dt.hour
    out["day_of_week"] = dt.dt.dayofweek
    out["month"] = dt.dt.month
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["day_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["day_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)

    t_cols = [f"T{i}" for i in range(1, 10) if f"T{i}" in out.columns]
    rh_cols = [f"RH_{i}" for i in range(1, 10) if f"RH_{i}" in out.columns]
    out["T_inside_mean"] = out[t_cols].mean(axis=1)
    out["RH_inside_mean"] = out[rh_cols].mean(axis=1)
    out["delta_T_out_inside"] = out["T_out"] - out["T_inside_mean"]

    out["Appliances_lag_1"] = out[TARGET].shift(1)
    out["Appliances_lag_3"] = out[TARGET].shift(3)
    out["Appliances_lag_6"] = out[TARGET].shift(6)
    past = out[TARGET].shift(1)
    out["Appliances_roll_3"] = past.rolling(3, min_periods=3).mean()
    out["Appliances_roll_6"] = past.rolling(6, min_periods=6).mean()
    out["Appliances_roll_12"] = past.rolling(12, min_periods=12).mean()
    return out


def prepare_dataset(csv_path: Path) -> pd.DataFrame:
    """Limpieza, features y drop de NaN por lag/rolling."""
    df = add_features(load_and_clean(csv_path))
    before = len(df)
    df = df.dropna().reset_index(drop=True)
    print(f"Filas tras lag/rolling: {len(df):,} (eliminadas: {before - len(df):,})")
    return df


def temporal_train_test_split(df: pd.DataFrame, test_ratio: float = TEST_RATIO):
    """Partición cronológica 80/20."""
    split_idx = int(len(df) * (1 - test_ratio))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


def get_xy(df: pd.DataFrame):
    y = df[TARGET]
    X = df.drop(columns=[TARGET, DATE_COL], errors="ignore")
    return X, y


def save_fig(name: str) -> None:
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figura guardada: {path}")

In [ ]:
df = prepare_dataset(DATA_PATH)
print(f'Registros: {len(df):,} | Variables: {df.shape[1]}')
print(f'Periodo: {df[DATE_COL].min()} a {df[DATE_COL].max()}')
df.head()

In [ ]:
df[[TARGET, 'lights', 'T_out', 'T_inside_mean', 'hour']].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df[DATE_COL], df[TARGET], linewidth=0.7)
ax.set_title('Consumo energético en el tiempo')
ax.set_xlabel('Fecha')
ax.set_ylabel('Consumo (Wh)')
plt.tight_layout()
save_fig('01_consumption_time.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[TARGET], bins=40, edgecolor='white')
ax.set_title('Distribución del consumo')
ax.set_xlabel('Consumo (Wh)')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
save_fig('02_consumption_histogram.png')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('hour')[TARGET].mean().plot(kind='bar', ax=axes[0])
axes[0].set_title('Consumo promedio por hora')
labels = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
daily = df.groupby('day_of_week')[TARGET].mean()
daily.index = [labels[i] for i in daily.index]
daily.plot(kind='bar', ax=axes[1])
axes[1].set_title('Consumo promedio por día de la semana')
plt.tight_layout()
save_fig('03_hour_weekday_means.png')
plt.show()

In [ ]:
corr_cols = [TARGET, 'lights', 'T_out', 'RH_out', 'hour',
             'T_inside_mean', 'RH_inside_mean', 'delta_T_out_inside',
             'Appliances_lag_1', 'Appliances_roll_6']
corr_cols = [c for c in corr_cols if c in df.columns]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de correlación')
plt.tight_layout()
save_fig('04_correlation_matrix.png')
plt.show()

### Notas del EDA
- Tras rezagos y medias móviles se eliminan filas iniciales con NaN.
- Hay patrones por **hora** y **día de la semana**.
- Continúa con el notebook **02** (split temporal y modelos).